#### Latest Time Stamp Win

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW latest_cdc AS
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY customer_id
               ORDER BY CAST(updated_datetime AS TIMESTAMP) DESC
           ) AS rn
    FROM associate_assignment.default.dim_customer_cdc
) sub
WHERE rn = 1;

In [0]:
%sql
select * from associate_assignment.default.dim_customer_cdc;

#### merge and cdc

In [0]:
%sql
-- I → Insert new customer
-- U → SCD Type 2 (expire old row + insert new row)
-- D → Expire the current record (instead of physically deleting, since we want history)

-- Step 1: Expire records for Updates and Deletes
MERGE INTO associate_assignment.default.dim_master_customer t
USING latest_cdc s
ON t.customer_id = s.customer_id
   AND t.current_flag = 'Y'

WHEN MATCHED
     AND s.operation = 'U'
     AND t.city <> s.city
THEN UPDATE SET
    t.current_flag = 'N',
    t.effective_to = current_timestamp(),
    t.updated_timestamp = s.updated_datetime

WHEN MATCHED
     AND s.operation = 'D'
THEN UPDATE SET
    t.current_flag = 'N',
    t.effective_to = current_timestamp(),
    t.updated_timestamp = s.updated_datetime;

In [0]:
%sql
-- Step 2: Insert New Customers and New Versions
INSERT INTO associate_assignment.default.dim_master_customer
(
    customer_id,
    name,
    city,
    status,
    _rescued_data,
    ingestion_timestamp,
    updated_timestamp,
    effective_from,
    effective_to,
    current_flag,
    source_file
)
SELECT
    s.customer_id,
    s.name,
    s.city,
    s.status,
    s._rescued_data,
    s.ingestion_timestamp,
    s.updated_datetime,
    current_timestamp(),
    '9999-12-31',
    'Y',
    s.source_file
FROM latest_cdc s
LEFT JOIN associate_assignment.default.dim_master_customer t
    ON s.customer_id = t.customer_id
   AND t.current_flag = 'Y'
WHERE
    (
        s.operation = 'I'
        AND t.customer_id IS NULL
    )
    OR
    (
        s.operation = 'U'
        AND (
            t.customer_id IS NULL
            OR t.city <> s.city
        )
    );